In [3]:
!pip install -q pandas numpy scikit-learn nltk spacy gensim matplotlib seaborn

In [2]:
!python -m spacy download pt_core_news_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/pt_core_news_sm-3.8.0/pt_core_news_sm-3.8.0-py3-none-any.whl (13.0 MB)
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [15]:
#Exercício 1 — Construção da Esteira de Pré-processamento


import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import spacy

from nltk.corpus import stopwords
from gensim.models import FastText
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 1. Preparação do ambiente

nltk.download("stopwords", quiet=True)

stop_words_pt = set(stopwords.words("portuguese"))

# Modelo de português do spaCy
nlp = spacy.load("pt_core_news_sm")

print("Ambiente preparado com sucesso.")


# 2. Função principal de pré-processamento

def limpar_e_lemmatizar(texto):
    """
    Realiza o pré-processamento de uma mensagem do SAC.

    Etapas:
    1. Converte para minúsculas
    2. Remove pontuação, números e caracteres especiais
    3. Tokeniza utilizando spaCy
    4. Remove stopwords
    5. Realiza lemmatization
    6. Remove tokens muito curtos
    7. Retorna o texto normalizado
    """

    # Garantir que a entrada seja uma string
    texto = str(texto)

    # 1. Converter para minúsculas
    texto_limpo = texto.lower()

    # 2. Remover pontuação, números e caracteres especiais
    texto_limpo = re.sub(
        r'[^a-záàâãéèêíïóôõöúçñ\s]',
        '',
        texto_limpo
    )

    # Remover espaços duplicados
    texto_limpo = re.sub(r'\s+', ' ', texto_limpo).strip()

    # 3. Processar o texto com spaCy
    doc = nlp(texto_limpo)

    # 4, 5 e 6. Filtrar tokens e realizar lemmatization
    tokens_filtrados = []

    for token in doc:

        # Ignorar espaços
        if token.is_space:
            continue

        # Ignorar stopwords
        if token.text in stop_words_pt:
            continue

        # Ignorar tokens muito pequenos
        if len(token.text) <= 1:
            continue

        # Obter o lema
        lema = token.lemma_.lower().strip()

        # Garantir que o lema não esteja vazio
        if lema and len(lema) > 1:
            tokens_filtrados.append(lema)

    # 7. Reconstruir a frase
    return " ".join(tokens_filtrados)


# 3. Função auxiliar para identificar os tokens antes/depois

def obter_tokens_originais(texto):
    """
    Prepara os tokens da mensagem antes da remoção
    de stopwords e lemmatization.
    """

    texto_limpo = str(texto).lower()

    texto_limpo = re.sub(
        r'[^a-záàâãéèêíïóôõöúçñ\s]',
        '',
        texto_limpo
    )

    texto_limpo = re.sub(r'\s+', ' ', texto_limpo).strip()

    doc = nlp(texto_limpo)

    tokens = []

    for token in doc:
        if token.is_space:
            continue

        if len(token.text) <= 1:
            continue

        tokens.append(token.text)

    return tokens


# 4. Função de diagnóstico

def analisar_preprocessamento(texto):
    """
    Apresenta um diagnóstico detalhado do pré-processamento.
    """

    # Tokens antes do processamento
    tokens_antes = obter_tokens_originais(texto)

    # Texto normalizado
    texto_normalizado = limpar_e_lemmatizar(texto)

    # Tokens depois do processamento
    tokens_depois = texto_normalizado.split()

    # Identificar tokens removidos
    tokens_finais_set = set(tokens_depois)

    tokens_removidos = [
        token
        for token in tokens_antes
        if token not in tokens_finais_set
    ]

    # Quantidade de caracteres
    quantidade_caracteres = len(texto)

    # Exibição do diagnóstico
    print("=" * 60)
    print("DIAGNÓSTICO DO PRÉ-PROCESSAMENTO")
    print("=" * 60)

    print("\nTexto original:")
    print(texto)

    print("\nTexto normalizado:")
    print(texto_normalizado)

    print("\nQuantidade de caracteres:")
    print(quantidade_caracteres)

    print("\nQuantidade de tokens antes:")
    print(len(tokens_antes))

    print("\nQuantidade de tokens depois:")
    print(len(tokens_depois))

    print("\nTokens removidos:")
    print(tokens_removidos)

    print("\nTokens finais:")
    print(tokens_depois)

    print("=" * 60)


# 5. Teste da função principal

frase_teste = (
    "Olá!!! EU gostaria de saber se vocês estão DEVOLVENDO "
    "as mesas que foram compradas ontem."
)

resultado = limpar_e_lemmatizar(frase_teste)

print("\nFrase original:")
print(frase_teste)

print("\nFrase processada:")
print(resultado)


# 6. Teste da função de diagnóstico


analisar_preprocessamento(
    "MEU sofá!!! chegou quebrado e quero DEVOLVER!!!"
)

Ambiente preparado com sucesso.

Frase original:
Olá!!! EU gostaria de saber se vocês estão DEVOLVENDO as mesas que foram compradas ontem.

Frase processada:
olá gostar saber devolver mesa comprar ontem
DIAGNÓSTICO DO PRÉ-PROCESSAMENTO

Texto original:
MEU sofá!!! chegou quebrado e quero DEVOLVER!!!

Texto normalizado:
sofá chegar quebrar querer devolver

Quantidade de caracteres:
47

Quantidade de tokens antes:
6

Quantidade de tokens depois:
5

Tokens removidos:
['meu', 'chegou', 'quebrado', 'quero']

Tokens finais:
['sofá', 'chegar', 'quebrar', 'querer', 'devolver']


In [24]:
!python dataset_sintetico_aula05.py



DATASETS DA MÓVEISDESIGN

Treinamento: 80 mensagens
Teste:       32 mensagens
OOD:         20 mensagens

Distribuição do treinamento:
intencao
logistica_entregas    20
trocas_devolucoes     20
vendas_orcamento      20
suporte_tecnico       20
Name: count, dtype: int64

Distribuição do teste:
intencao
vendas_orcamento      8
logistica_entregas    8
suporte_tecnico       8
trocas_devolucoes     8
Name: count, dtype: int64

Arquivos gerados:
✓ sac_moveis_ac2_treino.csv
✓ sac_moveis_ac2_teste.csv
✓ sac_moveis_ac2_ood.csv


In [40]:
# Exercício 2 — Construção da Representação Semântica com FastText + Mean Pooling


import pandas as pd
import numpy as np
from gensim.models import FastText


# 1. CARREGAR O DATASET

df = pd.read_csv("sac_moveis_ac2_ood.csv")

print("Dataset carregado!")
print(df.head())


# 2. APLICAR A FUNÇÃO DE LIMPEZA E LEMMATIZAÇÃO

df["mensagem_limpa"] = df["mensagem"].apply(limpar_e_lemmatizar)

print("\nMensagens após limpeza:")
print(df[["mensagem", "mensagem_limpa"]].head())


# 3. TRANSFORMAR CADA MENSAGEM EM LISTA DE TOKENS

corpus_tokenizado = []

for frase in df["mensagem_limpa"]:
    tokens = frase.split()
    corpus_tokenizado.append(tokens)

print("\nCorpus tokenizado:")
print(corpus_tokenizado[:5])


# 4. TREINAR O MODELO FASTTEXT

modelo_fasttext = FastText(
    sentences=corpus_tokenizado,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    sg=1
)

print("\nModelo FastText treinado com sucesso!")


# 5. FUNÇÃO DE MEAN POOLING


def obter_vetor_frase(frase, modelo):
    """
    Transforma uma frase em um vetor denso utilizando
    FastText + Mean Pooling.
    """

    palavras = frase.split()

    vetores = []

    for palavra in palavras:

        # TODO 1:
        # Obter o vetor da palavra no modelo FastText
        vetor = modelo.wv[palavra]

        # TODO 2:
        # Adicionar o vetor à lista
        vetores.append(vetor)

    # TODO 3:
    # Verificar se nenhum vetor foi encontrado
    if len(vetores) == 0:
        return np.zeros(modelo.vector_size)

    # TODO 4:
    # Calcular o Mean Pooling
    vetor_medio = np.mean(vetores, axis=0)

    return vetor_medio


# 6. TESTAR A FUNÇÃO

frase_teste = df["mensagem_limpa"].iloc[0]

vetor_frase = obter_vetor_frase(
    frase_teste,
    modelo_fasttext
)

print("\nFrase:")
print(frase_teste)

print("\nVetor da frase:")
print(vetor_frase)

print("\nDimensão do vetor:")
print(vetor_frase.shape)

Dataset carregado!
                                  mensagem         intencao
0  Qual é a previsão do tempo para amanhã?  fora_do_dominio
1            Quero aprender a tocar violão  fora_do_dominio
2        Me explique como funciona Bitcoin  fora_do_dominio
3     Quem ganhou o campeonato brasileiro?  fora_do_dominio
4    Qual será o clima no final de semana?  fora_do_dominio

Mensagens após limpeza:
                                  mensagem                mensagem_limpa
0  Qual é a previsão do tempo para amanhã?         previsão tempo amanhã
1            Quero aprender a tocar violão  querer aprender tocar violão
2        Me explique como funciona Bitcoin    explique funcionar bitcoin
3     Quem ganhou o campeonato brasileiro?  ganhar campeonato brasileiro
4    Qual será o clima no final de semana?            clima final semana

Corpus tokenizado:
[['previsão', 'tempo', 'amanhã'], ['querer', 'aprender', 'tocar', 'violão'], ['explique', 'funcionar', 'bitcoin'], ['ganhar', 'campeonato'

In [43]:
# Exercício 3 — Classificador de Intenções com Fallback


import pandas as pd
import numpy as np

from gensim.models import FastText
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report



# 1. CARREGAR DATASET DE TREINAMENTO

df = pd.read_csv("sac_moveis_ac2_treino.csv")

print("Dataset carregado!")
print(df.head())

print("\nDistribuição das intenções:")
print(df["intencao"].value_counts())


# 2. LIMPAR E LEMMATIZAR


df["mensagem_limpa"] = df["mensagem"].apply(
    limpar_e_lemmatizar
)


# 3. CRIAR CORPUS TOKENIZADO

corpus_tokenizado = [
    frase.split()
    for frase in df["mensagem_limpa"]
]


# 4. TREINAR FASTTEXT

modelo_fasttext = FastText(
    sentences=corpus_tokenizado,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    sg=1
)

print("\nFastText treinado!")


# 5. GERAR VETORES DAS FRASES

X_vetores = np.array([
    obter_vetor_frase(
        frase,
        modelo_fasttext
    )
    for frase in df["mensagem_limpa"]
])

y = df["intencao"].values

print("\nFormato de X_vetores:", X_vetores.shape)
print("Formato de y:", y.shape)


# 6. SEPARAR TREINO E TESTE


X_train, X_test, y_train, y_test = train_test_split(
    X_vetores,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nDados separados!")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


# 7. TREINAR REGRESSÃO LOGÍSTICA

modelo = LogisticRegression(max_iter=1000)

modelo.fit(X_train, y_train)

print("\nModelo treinado com sucesso!")


# 8. AVALIAR

y_pred = modelo.predict(X_test)

print("\n===== RELATÓRIO DE CLASSIFICAÇÃO =====")

print(
    classification_report(
        y_test,
        y_pred
    )
)


# 9. FALLBACK

def classificar_mensagem(
    mensagem,
    modelo,
    modelo_embedding,
    limiar=0.50
):

    # Limpar mensagem
    mensagem_limpa = limpar_e_lemmatizar(
        mensagem
    )

    # Transformar em vetor
    vetor = obter_vetor_frase(
        mensagem_limpa,
        modelo_embedding
    )

    # Transformar em matriz
    vetor = vetor.reshape(1, -1)

    # Probabilidades
    probabilidades = modelo.predict_proba(vetor)[0]

    # Maior probabilidade
    indice = np.argmax(probabilidades)

    # Intenção
    intencao = modelo.classes_[indice]

    # Confiança
    confianca = probabilidades[indice]

    # Fallback
    if confianca >= limiar:
        return intencao, confianca
    else:
        return "FALLBACK_HUMANO", confianca



# 10. TESTAR CHATBOT

testes = [
    "quero devolver meu sofá",
    "como faço para realizar a devolução?",
    "cadê meu pedido?",
    "meu pedido nao chego",
    "qual é a previsão do tempo?"
]


print("\n===== TESTES DO CHATBOT =====")

for mensagem in testes:

    intencao, confianca = classificar_mensagem(
        mensagem,
        modelo,
        modelo_fasttext
    )

    print("\nMensagem:", mensagem)
    print("Intenção:", intencao)
    print("Confiança:", f"{confianca:.2%}")

Dataset carregado!
                                            mensagem            intencao
0                           Meu pedido está atrasado  logistica_entregas
1      Quero devolver este sofá que chegou com rasgo   trocas_devolucoes
2       Qual o prazo de entrega do sofá que comprei?  logistica_entregas
3                Quero consultar o status da entrega  logistica_entregas
4  Recebi um armário com peças quebradas e quero ...   trocas_devolucoes

Distribuição das intenções:
intencao
logistica_entregas    20
trocas_devolucoes     20
vendas_orcamento      20
suporte_tecnico       20
Name: count, dtype: int64

FastText treinado!

Formato de X_vetores: (80, 50)
Formato de y: (80,)

Dados separados!
X_train: (64, 50)
X_test: (16, 50)

Modelo treinado com sucesso!

===== RELATÓRIO DE CLASSIFICAÇÃO =====
                    precision    recall  f1-score   support

logistica_entregas       1.00      0.75      0.86         4
   suporte_tecnico       0.75      0.75      0.75         4
 tr

In [45]:
# Exercício 4 — Laboratório Comparativo: Regressão Logística × KNN

import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# 1. UTILIZAR OS VETORES GERADOS NO EXERCÍCIO 3

X_train_vec = X_train
X_test_vec = X_test

print("X_train_vec:", X_train_vec.shape)
print("X_test_vec:", X_test_vec.shape)


# 2. CRIAR OS MODELOS

modelo_logistico = LogisticRegression(
    max_iter=1000
)

modelo_knn = KNeighborsClassifier(
    n_neighbors=3
)


# 3. TREINAR OS MODELOS

modelo_logistico.fit(
    X_train_vec,
    y_train
)

modelo_knn.fit(
    X_train_vec,
    y_train
)

print("\nModelos treinados com sucesso!")


# 4. FAZER PREVISÕES NOS DADOS DE TESTE

y_pred_logistico = modelo_logistico.predict(
    X_test_vec
)

y_pred_knn = modelo_knn.predict(
    X_test_vec
)


# 5. CALCULAR MÉTRICAS - REGRESSÃO LOGÍSTICA

accuracy_logistico = accuracy_score(
    y_test,
    y_pred_logistico
)

precision_logistico = precision_score(
    y_test,
    y_pred_logistico,
    average="weighted",
    zero_division=0
)

recall_logistico = recall_score(
    y_test,
    y_pred_logistico,
    average="weighted",
    zero_division=0
)

f1_logistico = f1_score(
    y_test,
    y_pred_logistico,
    average="weighted",
    zero_division=0
)


# 6. CALCULAR MÉTRICAS - KNN

accuracy_knn = accuracy_score(
    y_test,
    y_pred_knn
)

precision_knn = precision_score(
    y_test,
    y_pred_knn,
    average="weighted",
    zero_division=0
)

recall_knn = recall_score(
    y_test,
    y_pred_knn,
    average="weighted",
    zero_division=0
)

f1_knn = f1_score(
    y_test,
    y_pred_knn,
    average="weighted",
    zero_division=0
)


# 7. CRIAR TABELA COMPARATIVA

resultados = pd.DataFrame({
    "Modelo": [
        "Regressão Logística",
        "KNN"
    ],

    "Accuracy": [
        accuracy_logistico,
        accuracy_knn
    ],

    "Precision": [
        precision_logistico,
        precision_knn
    ],

    "Recall": [
        recall_logistico,
        recall_knn
    ],

    "F1": [
        f1_logistico,
        f1_knn
    ]
})


# 8. MOSTRAR RESULTADOS EM PORCENTAGEM

resultados_porcentagem = resultados.copy()

for coluna in [
    "Accuracy",
    "Precision",
    "Recall",
    "F1"
]:
    resultados_porcentagem[coluna] = (
        resultados_porcentagem[coluna] * 100
    ).round(2).astype(str) + "%"


print("\n==========================================")
print("COMPARAÇÃO DOS MODELOS")
print("==========================================")

print(resultados_porcentagem.to_string(index=False))

X_train_vec: (64, 50)
X_test_vec: (16, 50)

Modelos treinados com sucesso!

COMPARAÇÃO DOS MODELOS
             Modelo Accuracy Precision Recall     F1
Regressão Logística   68.75%    71.25% 68.75% 69.35%
                KNN    50.0%    41.67%  50.0% 43.33%
